In [13]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [1]:
import yfinance as yf
import pandas as pd
from pathlib import Path

aapl_5y = yf.download(
    "AAPL",
    start="2021-08-31",
    end="2026-09-01",
    interval="1d",
    auto_adjust=True,
    progress=False
)

print("Rows:", len(aapl_5y))
print("First date:", aapl_5y.index.min())
print("Last date:", aapl_5y.index.max())

display(aapl_5y.head())

Rows: 1254
First date: 2021-08-31 00:00:00
Last date: 2026-08-28 00:00:00


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2021-08-31,148.090607,149.036718,147.563898,148.900167,86453100
2021-09-01,148.753860,151.163029,148.588049,149.065986,80313700
2021-09-02,149.865768,150.909423,148.646555,150.080351,71115500
2021-09-03,150.499756,150.821630,149.319550,149.973047,57808700
2021-09-07,152.830902,153.386856,150.587546,151.153263,82278300


In [3]:
import time

tickers = ["AAPL", "MSFT", "NVDA", "QQQ"]

raw_5y_dir = Path("data/raw_5y_yahoo")
raw_5y_dir.mkdir(parents=True, exist_ok=True)

all_5y_data = []

for ticker in tickers:
    print(f"Downloading {ticker}...")

    df = yf.download(
        ticker,
        start="2021-08-31",
        end="2026-09-01",
        interval="1d",
        auto_adjust=True,
        progress=False
    )

    # Flatten yfinance multi-level columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df.insert(0, "Ticker", ticker)

    # Keep and reorder required daily OHLCV columns
    df = df[
        ["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]
    ]

    file_path = raw_5y_dir / f"{ticker}_daily_5y.csv"
    df.to_csv(file_path, index=False)

    print(f"{ticker}: {len(df)} rows saved to {file_path}")

    all_5y_data.append(df)
    time.sleep(1)

combined_5y_df = pd.concat(all_5y_data, ignore_index=True)

combined_file = raw_5y_dir / "all_tickers_daily_5y.csv"
combined_5y_df.to_csv(combined_file, index=False)

print("\nDownload complete.")
print("Combined rows:", len(combined_5y_df))
print("Combined file:", combined_file)

AAPL: 1254 rows saved to data\raw_5y_yahoo\AAPL_daily_5y.csv
MSFT: 1254 rows saved to data\raw_5y_yahoo\MSFT_daily_5y.csv
NVDA: 1254 rows saved to data\raw_5y_yahoo\NVDA_daily_5y.csv
QQQ: 1254 rows saved to data\raw_5y_yahoo\QQQ_daily_5y.csv

Download complete.
Combined rows: 5016
Combined file: data\raw_5y_yahoo\all_tickers_daily_5y.csv


In [5]:
print("=== Basic summary ===")

summary_5y = combined_5y_df.groupby("Ticker").agg(
    Rows=("Date", "size"),
    Start_Date=("Date", "min"),
    End_Date=("Date", "max"),
    Missing_Close=("Close", lambda x: x.isna().sum())
)

display(summary_5y)

print("\n=== Duplicate ticker-date rows ===")

duplicate_count_5y = combined_5y_df.duplicated(
    subset=["Ticker", "Date"]
).sum()

print(duplicate_count_5y)

print("\n=== Invalid OHLC rows ===")

invalid_ohlc_5y = combined_5y_df[
    (combined_5y_df["High"] <
     combined_5y_df[["Open", "Close", "Low"]].max(axis=1))
    |
    (combined_5y_df["Low"] >
     combined_5y_df[["Open", "Close", "High"]].min(axis=1))
]

print(len(invalid_ohlc_5y))

print("\n=== Missing values in all columns ===")

display(combined_5y_df.isna().sum())

=== Basic summary ===


,Rows,Start_Date,End_Date,Missing_Close
Ticker,,,,
AAPL,1254,2021-08-31,2026-08-28,1
MSFT,1254,2021-08-31,2026-08-28,1
NVDA,1254,2021-08-31,2026-08-28,1
QQQ,1254,2021-08-31,2026-08-28,1



=== Duplicate ticker-date rows ===
0

=== Invalid OHLC rows ===
0

=== Missing values in all columns ===


Price
Ticker    0
Date      0
Open      4
High      4
Low       4
Close     4
Volume    0
dtype: int64

In [7]:
missing_rows_5y = combined_5y_df[
    combined_5y_df[
        ["Open", "High", "Low", "Close", "Volume"]
    ].isna().any(axis=1)
]

print("Rows containing missing OHLCV values:")
display(missing_rows_5y)

Rows containing missing OHLCV values:


Price,Ticker,Date,Open,High,Low,Close,Volume
1253,AAPL,2026-08-28,NaN,NaN,NaN,NaN,38623623
2507,MSFT,2026-08-28,NaN,NaN,NaN,NaN,29189683
3761,NVDA,2026-08-28,NaN,NaN,NaN,NaN,194792961
5015,QQQ,2026-08-28,NaN,NaN,NaN,NaN,34067577


In [9]:
# Preserve the original downloaded data
cleaned_5y_df = combined_5y_df.dropna(
    subset=["Open", "High", "Low", "Close", "Volume"]
).copy()

cleaned_5y_df = cleaned_5y_df.sort_values(
    ["Date", "Ticker"]
).reset_index(drop=True)

print("Original rows:", len(combined_5y_df))
print("Cleaned rows:", len(cleaned_5y_df))
print("Removed rows:", len(combined_5y_df) - len(cleaned_5y_df))

display(
    cleaned_5y_df.groupby("Ticker").agg(
        Rows=("Date", "size"),
        Start_Date=("Date", "min"),
        End_Date=("Date", "max")
    )
)

Original rows: 5016
Cleaned rows: 5012
Removed rows: 4


,Rows,Start_Date,End_Date
Ticker,,,
AAPL,1253,2021-08-31,2026-08-27
MSFT,1253,2021-08-31,2026-08-27
NVDA,1253,2021-08-31,2026-08-27
QQQ,1253,2021-08-31,2026-08-27


In [11]:
print("=== Number of tickers available on each date ===")

date_counts_5y = cleaned_5y_df.groupby("Date")["Ticker"].nunique()
display(date_counts_5y.value_counts().sort_index())

incomplete_dates_5y = date_counts_5y[date_counts_5y != 4]

print("\nDates without all four tickers:", len(incomplete_dates_5y))

if len(incomplete_dates_5y) > 0:
    display(incomplete_dates_5y)
else:
    print("All dates contain AAPL, MSFT, NVDA and QQQ.")

=== Number of tickers available on each date ===


Ticker
4    1253
Name: count, dtype: int64


Dates without all four tickers: 0
All dates contain AAPL, MSFT, NVDA and QQQ.


In [13]:
processed_5y_dir = Path("data/processed_5y_yahoo")
processed_5y_dir.mkdir(parents=True, exist_ok=True)

aligned_5y_file = processed_5y_dir / "aligned_daily_ohlcv_5y.csv"

cleaned_5y_df.to_csv(
    aligned_5y_file,
    index=False
)

print("Saved:", aligned_5y_file)
print("Rows:", len(cleaned_5y_df))
print("Columns:", len(cleaned_5y_df.columns))

display(cleaned_5y_df.head(8))

Saved: data\processed_5y_yahoo\aligned_daily_ohlcv_5y.csv
Rows: 5012
Columns: 7


Price,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2021-08-31,148.900167,149.036718,147.563898,148.090607,86453100
1,MSFT,2021-08-31,292.160127,292.236892,289.357711,289.722412,26285300
2,NVDA,2021-08-31,22.620757,22.620757,22.047639,22.311771,259850000
3,QQQ,2021-08-31,369.280898,369.348810,367.194323,368.737427,29628200
4,AAPL,2021-09-01,149.065971,151.163013,148.588034,148.753845,80313700
5,MSFT,2021-09-01,290.672594,292.899168,289.348165,289.674469,18983800
6,NVDA,2021-09-01,22.411449,22.622755,22.283867,22.367592,201767000
7,QQQ,2021-09-01,369.795232,371.415932,369.144991,369.348785,28138300


In [15]:
cleaned_5y_df.columns.name = None
cleaned_5y_df.to_csv(aligned_5y_file, index=False)

display(cleaned_5y_df.head())

,Ticker,Date,Open,High,Low,Close,Volume
0,AAPL,2021-08-31,148.900167,149.036718,147.563898,148.090607,86453100
1,MSFT,2021-08-31,292.160127,292.236892,289.357711,289.722412,26285300
2,NVDA,2021-08-31,22.620757,22.620757,22.047639,22.311771,259850000
3,QQQ,2021-08-31,369.280898,369.348810,367.194323,368.737427,29628200
4,AAPL,2021-09-01,149.065971,151.163013,148.588034,148.753845,80313700


In [3]:
# Final audit: reload files from disk

import pandas as pd
from pathlib import Path

aligned_5y_file = Path(
    "data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv"
)
raw_files = {
    "AAPL": Path("data/raw_5y_yahoo/AAPL_daily_5y.csv"),
    "MSFT": Path("data/raw_5y_yahoo/MSFT_daily_5y.csv"),
    "NVDA": Path("data/raw_5y_yahoo/NVDA_daily_5y.csv"),
    "QQQ": Path("data/raw_5y_yahoo/QQQ_daily_5y.csv")
}

print("=== RAW FILE AUDIT ===")

for ticker, file_path in raw_files.items():
    raw_df = pd.read_csv(file_path)

    missing_ohlc = raw_df[
        ["Open", "High", "Low", "Close"]
    ].isna().any(axis=1).sum()

    duplicates = raw_df.duplicated(
        subset=["Ticker", "Date"]
    ).sum()

    print(
        f"{ticker}: "
        f"rows={len(raw_df)}, "
        f"missing_OHLC_rows={missing_ohlc}, "
        f"duplicates={duplicates}"
    )

print("\n=== PROCESSED FILE AUDIT ===")

processed_check = pd.read_csv(aligned_5y_file)

print("Rows:", len(processed_check))
print("Columns:", list(processed_check.columns))
print("Start date:", processed_check["Date"].min())
print("End date:", processed_check["Date"].max())
print("Missing values:", processed_check.isna().sum().sum())
print(
    "Duplicate ticker-date rows:",
    processed_check.duplicated(["Ticker", "Date"]).sum()
)

print("\nRows per ticker:")
display(processed_check["Ticker"].value_counts().sort_index())

=== RAW FILE AUDIT ===
AAPL: rows=1254, missing_OHLC_rows=1, duplicates=0
MSFT: rows=1254, missing_OHLC_rows=1, duplicates=0
NVDA: rows=1254, missing_OHLC_rows=1, duplicates=0
QQQ: rows=1254, missing_OHLC_rows=1, duplicates=0

=== PROCESSED FILE AUDIT ===
Rows: 5012
Columns: ['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']
Start date: 2021-08-31
End date: 2026-08-27
Missing values: 0
Duplicate ticker-date rows: 0

Rows per ticker:


Ticker
AAPL    1253
MSFT    1253
NVDA    1253
QQQ     1253
Name: count, dtype: int64

In [5]:
data_note = """# Data Documentation

## Overview

This dataset contains daily market data for AAPL, MSFT, NVDA, and QQQ. It is intended for subsequent feature construction, target creation, and modelling experiments.

## Data Source

- Source: Yahoo Finance
- Download tool: Python `yfinance`
- Download interval: Daily (`1d`)
- Requested period: 2021-08-31 to 2026-09-01
- Price adjustment: `auto_adjust=True`
- No API key is required.

Because `auto_adjust=True` was used, the Open, High, Low, and Close columns contain adjusted prices.

## Columns

The raw and processed datasets use the following columns:

- `Ticker`: Instrument symbol
- `Date`: Trading date in `YYYY-MM-DD` format
- `Open`: Adjusted opening price
- `High`: Adjusted highest price
- `Low`: Adjusted lowest price
- `Close`: Adjusted closing price
- `Volume`: Daily trading volume

## Raw Data

The four raw files are:

- `AAPL_daily_5y.csv`
- `MSFT_daily_5y.csv`
- `NVDA_daily_5y.csv`
- `QQQ_daily_5y.csv`

The raw files have been preserved without modification. Each raw file contains 1,254 rows covering 2021-08-31 to 2026-08-28.

## Data Quality Checks

The following checks were performed:

- Missing-value check
- Duplicate ticker-date check
- OHLC consistency check
- Chronological sorting
- Common trading-date alignment across all four instruments
- Column consistency check

No duplicate ticker-date records or invalid OHLC relationships were found.

## Removed Records

Four incomplete records dated 2026-08-28 were removed from the processed dataset:

- AAPL — 2026-08-28
- MSFT — 2026-08-28
- NVDA — 2026-08-28
- QQQ — 2026-08-28

These records contained Volume values, but Open, High, Low, and Close were all missing. The values were not imputed because doing so would create artificial price data.

The original records remain unchanged in the raw files.

## Final Processed Dataset

File:

`data/processed_5y_yahoo/aligned_daily_ohlcv_5y.csv`

Final dataset details:

- Date range: 2021-08-31 to 2026-08-27
- Rows per instrument: 1,253
- Total rows: 5,012
- Missing values: 0
- Duplicate ticker-date rows: 0
- All four instruments use the same 1,253 common trading dates
- Data is sorted chronologically by Date and Ticker

This processed dataset should be used as the shared input for the next feature and modelling stage.

## Reproducibility

The accompanying Jupyter Notebook contains the download, validation, cleaning, alignment, and export process. No API key or credential is stored in the notebook.
"""

readme_file = Path("DATA_README.md")
readme_file.write_text(data_note, encoding="utf-8")

print("Created:", readme_file)
print("Characters:", len(data_note))

Created: DATA_README.md
Characters: 2478
